<a href="https://colab.research.google.com/github/Chediak/common-master-ai/blob/main/semeq_desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q tensorflow numpy pandas scikit-learn scipy matplotlib kaggle kagglehub pywavelets imbalanced-learn seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 41.9 MB/s eta 0:00:00


In [4]:
import os, numpy as np, pandas as pd, glob, tensorflow as tf, pywt, seaborn as sns, matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential; from tensorflow.keras.layers import Dense, LSTM, Conv1D, Dropout, BatchNormalization, Bidirectional, Input, Flatten
from tensorflow.keras.regularizers import l2; from tensorflow.keras.optimizers import Adam; from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split; from sklearn.preprocessing import StandardScaler; from sklearn.metrics import confusion_matrix
from collections import Counter
from tqdm import tqdm

print("Initializing...")

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("TPU enabled!")
except:
    strategy = tf.distribute.get_strategy()
    print("Running on:", "GPU" if tf.config.list_physical_devices('GPU') else "CPU")


print("Downloading dataset...")
dataset_path = "/root/.cache/kagglehub/datasets/uysalserkan/fault-induction-motor-dataset/versions/1"
if not os.path.exists(dataset_path):
    try:
        import kagglehub
        dataset_path = kagglehub.dataset_download("uysalserkan/fault-induction-motor-dataset")
    except:
        print("KaggleHub failed. Download manually.")

print("Loading dataset...")
normal_files = glob.glob(dataset_path + "/normal/normal/*.csv")
imbalances = [glob.glob(dataset_path + f"/imbalance/imbalance/{g}/*.csv") for g in ["6g", "10g", "15g", "20g", "25g", "30g"]]

print("Processing data...")
dataReader = lambda file_paths: pd.concat([pd.read_csv(f, header=None) for f in tqdm(file_paths, desc="Reading CSVs")], ignore_index=True) if file_paths else pd.DataFrame()
extract_features = lambda data: pd.DataFrame({'mean': data.mean(axis=1), 'std': data.std(axis=1), 'skew': data.skew(axis=1), 'kurtosis': data.kurtosis(axis=1)}) if not data.empty else pd.DataFrame()
wavelet_transform = lambda data: pd.DataFrame(np.concatenate(pywt.wavedec(data.to_numpy(), 'db1', level=2, axis=0), axis=0)) if not data.empty else pd.DataFrame()

print("Extracting features...")
data_n, imbalances = dataReader(normal_files), [dataReader(paths) for paths in tqdm(imbalances, desc="Reading imbalance data")]
data_n, imbalances = extract_features(data_n), [extract_features(data) for data in tqdm(imbalances, desc="Extracting features")]
data_n, imbalances = wavelet_transform(data_n), [wavelet_transform(data) for data in tqdm(imbalances, desc="Applying Wavelet Transform")]

print("Preparing training data...")
X = pd.concat([data_n] + imbalances, ignore_index=True)
y = pd.concat([pd.DataFrame(np.full((len(data), 1), i)) for i, data in enumerate([data_n] + imbalances)], ignore_index=True)
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
X = StandardScaler().fit_transform(X)

print("Splitting dataset into train and test...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=True)

y_train = to_categorical(y_train, num_classes=len(np.unique(y)))
y_test = to_categorical(y_test, num_classes=len(np.unique(y)))

X_train, X_test = np.expand_dims(X_train, axis=-1), np.expand_dims(X_test, axis=-1)

print("Building model...")
with strategy.scope():
    model = Sequential([
        Input(shape=(X.shape[1], 1)),
        Conv1D(64, 3, activation='relu', kernel_regularizer=l2(0.01)), BatchNormalization(), Dropout(0.4),
        Bidirectional(LSTM(32, return_sequences=False)),
        Dense(32, activation='relu', kernel_regularizer=l2(0.01)), Dense(len(np.unique(y)), activation='softmax')])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

print("Training model...")
batch_size = 32 if tf.config.list_physical_devices('GPU') else 16
history = model.fit(X_train, y_train, epochs=30, validation_split=0.2, batch_size=batch_size,
                    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True), ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)],
                    verbose=1)

print("Evaluating model...")
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

print("Generating predictions...")
y_pred_classes = np.argmax(model.predict(X_test), axis=1)
y_test_labels = np.argmax(y_test, axis=1)
conf_matrix = confusion_matrix(y_test_labels, y_pred_classes)

print("Plotting confusion matrix...")
plt.figure(figsize=(8,6)); sns.heatmap(conf_matrix, annot=True, fmt='d', cmap="Blues"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - Optimized Model"); plt.show()

print("Plotting training history...")
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.plot(history.history['accuracy'], label='Train'); plt.plot(history.history['val_accuracy'], label='Validation'); plt.title('Model Accuracy'); plt.ylabel('Accuracy'); plt.xlabel('Epoch'); plt.legend(loc='lower right')
plt.subplot(1, 2, 2); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Validation'); plt.title('Model Loss'); plt.ylabel('Loss'); plt.xlabel('Epoch'); plt.legend(loc='upper right'); plt.show()

print("Completed")

Initializing...
Running on: GPU
Loading dataset...
Processing data...
Extracting features...


Applying Wavelet Transform: 100%|██████████| 6/6 [00:08<00:00,  1.49s/it]


Preparing training data...
Splitting dataset into train and test...
Building model...
Training model...
Epoch 1/30
 477020/1579688 ━━━━━━━━━━━━━━━━━━━━ 2:11:11 7ms/step - accuracy: 0.1862 - loss: 1.9145

KeyboardInterrupt: 

In [ ]:
import os, numpy as np, pandas as pd, glob, tensorflow as tf, pywt, seaborn as sns, matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential; from tensorflow.keras.layers import Dense, LSTM, Conv1D, Dropout, BatchNormalization, Bidirectional, Input
from tensorflow.keras.regularizers import l2; from tensorflow.keras.optimizers import Adam; from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split; from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from tqdm import tqdm; from collections import Counter

print("Initializing...")

try: tpu = tf.distribute.cluster_resolver.TPUClusterResolver(); tf.config.experimental_connect_to_cluster(tpu); tf.tpu.experimental.initialize_tpu_system(tpu); strategy = tf.distribute.TPUStrategy(tpu); print("✅ TPU enabled!")
except: strategy = tf.distribute.get_strategy(); print("Running on:", "GPU" if tf.config.list_physical_devices('GPU') else "CPU")

print("Downloading dataset...")
dataset_path = "/root/.cache/kagglehub/datasets/uysalserkan/fault-induction-motor-dataset/versions/1"
if not os.path.exists(dataset_path):
    try: import kagglehub; dataset_path = kagglehub.dataset_download("uysalserkan/fault-induction-motor-dataset")
    except: print("KaggleHub failed. Download manually.")

print("Loading dataset...")
normal_files = glob.glob(dataset_path + "/normal/normal/*.csv")
imbalances = [glob.glob(dataset_path + f"/imbalance/imbalance/{g}/*.csv") for g in ["6g", "10g", "15g", "20g", "25g", "30g"]]

print("Processing data...")
dataReader = lambda file_paths: pd.concat([pd.read_csv(f, header=None) for f in tqdm(file_paths, desc="Reading CSVs")], ignore_index=True) if file_paths else pd.DataFrame()
extract_features = lambda data: pd.DataFrame({'mean': data.mean(axis=1), 'std': data.std(axis=1), 'skew': data.skew(axis=1), 'kurtosis': data.kurtosis(axis=1)}) if not data.empty else pd.DataFrame()
wavelet_transform = lambda data: pd.DataFrame(np.concatenate(pywt.wavedec(data.to_numpy(), 'db1', level=2, axis=0), axis=0)) if not data.empty else pd.DataFrame()

print("Extracting features...")
data_n, imbalances = dataReader(normal_files), [dataReader(paths) for paths in tqdm(imbalances, desc="Reading imbalance data")]
data_n, imbalances = extract_features(data_n), [extract_features(data) for data in tqdm(imbalances, desc="Extracting features")]
data_n, imbalances = wavelet_transform(data_n), [wavelet_transform(data) for data in tqdm(imbalances, desc="Applying Wavelet Transform")]

print("Preparing training data...")
X = pd.concat([data_n] + imbalances, ignore_index=True)
y = pd.concat([pd.DataFrame(np.full((len(data), 1), i)) for i, data in enumerate([data_n] + imbalances)], ignore_index=True)
X = X.apply(pd.to_numeric, errors='coerce').fillna(0); X = StandardScaler().fit_transform(X)

print("Splitting dataset into train and test...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=True)
X_train, X_test = np.expand_dims(X_train, axis=-1), np.expand_dims(X_test, axis=-1)

y_train = to_categorical(y_train, num_classes=len(np.unique(y)))
y_test = to_categorical(y_test, num_classes=len(np.unique(y)))

print("Building model...")
with strategy.scope():
    model = Sequential([
        Input(shape=(X.shape[1], 1)),
        Conv1D(64, 3, activation='relu', kernel_regularizer=l2(0.01)), BatchNormalization(), Dropout(0.4),
        Bidirectional(LSTM(32, return_sequences=False)), Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
        Dense(len(np.unique(y)), activation='softmax')])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

print("Training model...")
batch_size = 32 if tf.config.list_physical_devices('GPU') else 16
history = model.fit(X_train, y_train, epochs=30, validation_split=0.2, batch_size=batch_size,
                    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True), ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)],
                    verbose=1)

print("Evaluating model...")
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

print("Generating predictions...")
y_pred_classes = np.argmax(model.predict(X_test), axis=1)
y_test_labels = np.argmax(y_test, axis=1)
conf_matrix = confusion_matrix(y_test_labels, y_pred_classes)

precision = precision_score(y_test_labels, y_pred_classes, average='weighted')
recall = recall_score(y_test_labels, y_pred_classes, average='weighted')
f1 = f1_score(y_test_labels, y_pred_classes, average='weighted')
print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1-score: {f1:.4f}")

history.history['precision'] = [precision] * len(history.history['accuracy'])
history.history['recall'] = [recall] * len(history.history['accuracy'])
history.history['f1_score'] = [f1] * len(history.history['accuracy'])

print("Plotting results...")
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.plot(history.history['accuracy'], label='Train'); plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy'); plt.ylabel('Accuracy'); plt.xlabel('Epoch'); plt.legend(loc='lower right')

plt.subplot(1, 2, 2); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss'); plt.ylabel('Loss'); plt.xlabel('Epoch'); plt.legend(loc='upper right'); plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history.history['precision'], label='Precision'); plt.plot(history.history['recall'], label='Recall'); plt.plot(history.history['f1_score'], label='F1-score')
plt.title('Precision, Recall, and F1-score'); plt.xlabel('Epoch'); plt.ylabel('Score'); plt.legend(loc='lower right'); plt.show()

plt.figure(figsize=(8,6)); sns.heatmap(conf_matrix, annot=True, fmt='d', cmap="Blues"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.show()

print("Completed]")


In [ ]:
!ls -R /root/.cache/kagglehub/datasets/uysalserkan/fault-induction-motor-dataset/versions/1